# Exception Handling

## 🎯 Learning Goal

By the end of this note, I will understand what an exception actually is, why unhandled exceptions crash a program, how `try`/`except`/`else`/`finally` work together, how to catch specific exception types (and why catching everything with a bare `except:` is risky), how to `raise` your own exceptions (including custom exception classes), and how Python's built-in exceptions are organized into a hierarchy.

---

## 🤔 What is it?

An **exception** is an unexpected event that happens *while a program is running* and disrupts its normal flow — dividing by zero, opening a file that doesn't exist, converting `"abc"` to a number. Python doesn't just silently ignore this: it **raises** an exception object, and unless something in the code catches it, the program **crashes** and prints a traceback.

> 🧸 Think of an exception like a fire alarm 🔥 going off in a building. The alarm (the exception) doesn't fix the fire — it just interrupts everything and says "something is wrong, right here, right now." What happens next depends entirely on whether anyone is trained to respond (a `try`/`except` block) or whether everyone just stands there while the building burns down (an unhandled exception — the program crashes).

---

## ❓ Why do we need it?

- Without exception handling, ANY unexpected problem — bad user input, a missing file, a network timeout — **immediately stops the entire program**, right at that line, no matter how much useful work was already done.
- Real programs need to **anticipate things going wrong** and decide what to do about it: show a friendly error message, try a fallback, log the problem and continue, or clean up resources (closing a file, releasing a lock) before giving up.
- `try`/`except` lets you separate "the code that might fail" from "what to do if it fails" — instead of littering every line with manual `if` checks for every possible way it could go wrong.

---

## 🧠 Key Idea — The Mental Model: `try` / `except`

- Put the risky code inside a `try:` block. Python runs it normally, line by line, **unless** an exception is raised partway through.
- The moment an exception is raised inside `try`, Python **immediately jumps out** of the rest of the `try` block — no more lines in it run — and looks for a matching `except:` block.
- If a matching `except` is found, its code runs, and the program **keeps going** afterward, as if nothing crashed. If no matching `except` exists anywhere, the exception keeps traveling upward until it reaches the top of the program and crashes it, printing a traceback.
- `else` (optional) runs only if the `try` block finished with **no** exception at all. `finally` (optional) always runs — exception or not — making it the right place for cleanup code.
- You can `raise` your own exceptions too — including ones from custom classes you write — to signal "this specific situation is invalid" in your own code, not just when Python itself hits a problem.

---

## 📚 Important Terms

| Term | Simple Meaning | Example |
|------|----------------|----------|
| Exception | An object Python creates to represent something going wrong at runtime | `ZeroDivisionError`, `ValueError` |
| Raise | To trigger an exception, either by Python itself or by your own code | `raise ValueError("bad input")` |
| `try` block | The section of code that might fail | `try: risky_code()` |
| `except` block | The code that runs IF a matching exception is raised in `try` | `except ValueError as e:` |
| `else` (with try) | Runs only if `try` succeeded with NO exception | `else: print("all good")` |
| `finally` | Always runs, exception or not — used for cleanup | `finally: file.close()` |
| Traceback | The error report Python prints when an exception goes unhandled | `ZeroDivisionError: division by zero` |
| Custom Exception | Your own exception class, usually inheriting from `Exception` | `class InsufficientBalanceError(Exception):` |
| Exception Chaining | Linking a new exception to the original one that caused it | `raise RuntimeError(...) from original_error` |

---

## 🔄 How it Works

```mermaid
flowchart TD
Try["try: risky_code()"] -->|No error| Else["else: (optional) runs"]
Try -->|Error raised| Except["except SomeError as e: handle it"]
Else --> Finally["finally: (optional) always runs"]
Except --> Finally
Finally --> Continue["Program keeps running normally"]
```

Read it like this: Python always attempts the `try` block first. If it succeeds cleanly, `else` runs (if present). If it fails, Python looks for a matching `except`. Either way — success or handled failure — `finally` runs at the very end, and then the program continues past the whole `try` statement instead of crashing.

---

## 🌍 Real-Life Example

Think of a **waiter at a restaurant** 🍽️ carrying a tray of food. The "risky" action is carrying the tray to the table (the `try` block) — it *usually* works, but the waiter could trip. If they trip (an exception), a trained response kicks in immediately: apologize, clean up, offer a replacement dish (the `except` block) — instead of the whole restaurant shutting down over one dropped tray. If the tray makes it safely, everyone's happy (`else`). Either way, the waiter always goes back to the kitchen to report what happened (`finally`) — successful delivery or accident, that step never gets skipped.

---

## 💻 First, See What Happens WITHOUT Exception Handling

Here's the crash in action — no `try`/`except` at all. The moment `10 / 0` raises `ZeroDivisionError`, Python stops executing immediately: `print("This line never runs")` genuinely never runs, and Python prints a traceback instead.

In [ ]:
result = 10 / 0
print("This line never runs")


**Note:** This cell **intentionally errors** — it's demonstrating exactly what happens when nothing catches the exception. `ZeroDivisionError: division by zero` is printed as a traceback, and the program stops right there — `print("This line never runs")` is never reached.

---

## 💻 Now, the Fix — a Basic `try` / `except`

Wrapping the risky line in `try:` and adding a matching `except:` lets the program catch the problem and keep running instead of crashing.

In [ ]:
try:
    result = 10 / 0
except ZeroDivisionError as e:
    print(f"Caught an exception: {e}")

print("The program kept running -- this line DOES print now!")


**Note:** `except ZeroDivisionError as e:` catches ONLY a `ZeroDivisionError` — `as e` gives you access to the actual exception object, letting you read its message (`str(e)` → `"division by zero"`) instead of just knowing *that* something went wrong. Compare this to the crash above: this time, execution continues normally right after the `try`/`except` block finishes.

---

## 🎯 Catching Specific Exception Types (Multiple `except` Blocks)

Different problems usually need different responses. A function can have several `except` blocks, each catching a different exception type — Python checks them top to bottom and runs the first one that matches.

In [ ]:
def divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        print("Cannot divide by zero!")
    except TypeError:
        print("Both arguments must be numbers!")

print(divide(10, 2))     # works fine -- returns 5.0
print(divide(10, 0))     # ZeroDivisionError -- caught by the first except
print(divide(10, "a"))   # TypeError -- caught by the second except


**Note:** `divide(10, 0)` and `divide(10, "a")` both fail, but for genuinely DIFFERENT reasons — dividing by zero vs. trying to divide by a string. Because each is caught by its own specific `except`, the error message can be specific and helpful too. Notice `divide()` doesn't explicitly `return` anything inside its `except` blocks, so those calls print `None` — that's a real, common beginner trap: catching an exception doesn't automatically give you a sensible return value, you still have to provide one if the function needs to return something even in the failure case.

**A shortcut — catching several types with ONE `except`:** if two exceptions should be handled the exact same way, list them together in a tuple instead of repeating the same handler twice.

In [ ]:
try:
    value = int("abc")
except (ValueError, TypeError) as e:   # one except, TWO exception types in a tuple
    print(f"Caught: {type(e).__name__}: {e}")


**Note:** `except (ValueError, TypeError) as e:` catches either one with the SAME handler — `int("abc")` happens to raise `ValueError` specifically, but this pattern is for when you genuinely want to treat several exception types identically.

---

## ⚠️ The Bare `except:` Trap

You CAN write a plain `except:` with no exception type at all — it catches **literally everything**, including exceptions you never intended to catch, like a typo in your own code. This usually hides real bugs instead of handling them.

In [ ]:
def risky_calculation(numbers):
    try:
        total = sum(numbers)
        avg = total / lenn(numbers)   # typo: "lenn" instead of "len" -- a real bug!
        return avg
    except:   # bare except -- catches EVERYTHING, including the typo's NameError above
        print("Something went wrong, returning 0")
        return 0

print(risky_calculation([1, 2, 3]))


**Bug found:** `lenn(numbers)` is a typo — there's no function called `lenn`, so this raises `NameError: name 'lenn' is not defined`. But because the `except:` is bare (no exception type), it silently swallows the `NameError` too, printing the generic "Something went wrong" message and returning `0` — hiding a real, fixable bug behind a message that sounds like a normal, expected failure. If this had used `except ZeroDivisionError:` (the error it was probably written to catch), the typo would have crashed loudly and been caught during testing instead of silently returning wrong data in production.

> ❌ Misconception: a bare `except:` is just a convenient way to "catch everything, just in case."
> ✅ Correct: it catches things you never meant to catch too — typos, `KeyboardInterrupt`, programming mistakes — making real bugs indistinguishable from expected failure cases. Always catch the SPECIFIC exception(s) you're actually expecting, or at most `except Exception:` (which still excludes things like `KeyboardInterrupt` and `SystemExit`).

---

## 🧩 `else` and `finally` — Completing the Picture

`else` runs only when `try` succeeds completely with no exception. `finally` runs **no matter what** — success, handled failure, or even an unhandled one — making it the right place for cleanup that must always happen.

In [ ]:
def safe_divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError:
        print("Error: division by zero")
    else:
        print(f"Result: {result}")   # only runs if NO exception happened
    finally:
        print("Done attempting division")   # ALWAYS runs, no matter what

safe_divide(10, 2)   # succeeds -- else AND finally both run
print("---")
safe_divide(10, 0)   # fails -- except AND finally both run, else is skipped


**Note:** `else` isn't just "put the success code here instead of at the end of `try`" — it specifically means "ONLY if nothing in `try` raised an exception." Putting `print(f"Result: {result}")` directly inside `try` instead would ALSO catch any exception `print()` itself might raise, which `else` deliberately avoids. `finally` proves it runs unconditionally: `"Done attempting division"` prints both times, regardless of whether the division succeeded or failed.

> ❌ Misconception: `finally` only runs if the exception was actually caught by an `except`.
> ✅ Correct: `finally` runs even if the exception is NEVER caught at all (it still crashes afterward, but `finally` gets to run first) — this is exactly why it's the right place to close a file or release a lock, guaranteeing cleanup happens either way.

---

## 🚀 `raise` — Triggering Your Own Exceptions

So far, Python has been raising exceptions automatically (dividing by zero, bad conversions). You can also `raise` one yourself, on purpose, to reject invalid input or an invalid state in YOUR OWN code — before it causes a more confusing failure somewhere else.

In [ ]:
def set_age(age):
    if age < 0:
        raise ValueError("Age cannot be negative")
    return age

try:
    set_age(-5)
except ValueError as e:
    print(f"Caught: {e}")

print(set_age(25))   # valid input -- no exception, works normally


**Note:** `raise ValueError("Age cannot be negative")` creates and triggers a brand-new exception, exactly like Python itself would for a bad conversion. `set_age(-5)` immediately stops at the `raise` line — the `return age` on the next line never runs for invalid input. Using an existing type like `ValueError` here is reasonable — the age *is* a genuinely invalid value — but sometimes you want something more specific to your own program. That's where custom exception classes come in.

---

## 🏗️ Custom Exceptions — Your Own Exception Class

Built-in exceptions (`ValueError`, `TypeError`, ...) are generic. A **custom exception** — a class you write, inheriting from `Exception` — lets you name the EXACT problem in your domain, and even attach extra data to it (like the account balance in the example below).

In [ ]:
class InsufficientBalanceError(Exception):   # inherits from Exception, like every custom exception should
    def __init__(self, balance, amount):
        self.balance = balance
        self.amount = amount
        super().__init__(f"Cannot withdraw {amount}, balance is only {balance}")

class BankAccount:
    def __init__(self, balance):
        self.balance = balance

    def withdraw(self, amount):
        if amount > self.balance:
            raise InsufficientBalanceError(self.balance, amount)
        self.balance -= amount
        return self.balance


acc = BankAccount(1000)

try:
    acc.withdraw(5000)
except InsufficientBalanceError as e:
    print(f"Transaction failed: {e}")
    print(f"Attempted: {e.amount}, Available: {e.balance}")   # the EXTRA data our custom exception carries


**Note:** `InsufficientBalanceError(Exception)` — inheriting from `Exception` is what makes this a real, catchable exception class at all; without it, `raise InsufficientBalanceError(...)` wouldn't work. `super().__init__(...)` still sets the normal exception message (so `str(e)` and `print(e)` work as expected), while `self.balance` and `self.amount` add EXTRA structured data most built-in exceptions don't offer — the code catching it can read `e.amount` and `e.balance` directly, instead of trying to parse numbers back out of a plain string message.

> ❌ Misconception: custom exceptions need to override lots of methods to work properly.
> ✅ Correct: inheriting from `Exception` and writing `__init__` (calling `super().__init__(...)` for the message) is usually all you need — everything else (being catchable, printable, having a traceback) comes for free from `Exception`.

---

## 🔗 Exception Chaining — `raise ... from ...`

Sometimes an exception happens WHILE handling another one. `raise NewError(...) from original_error` explicitly links the two, so anyone debugging later can see the real root cause, not just the last thing that went wrong.

In [ ]:
def load_config():
    try:
        with open("nonexistent_config.txt") as f:
            return f.read()
    except FileNotFoundError as e:
        raise RuntimeError("Could not load configuration") from e   # chains the new error to the original


try:
    load_config()
except RuntimeError as e:
    print(f"RuntimeError: {e}")
    print(f"Caused by: {type(e.__cause__).__name__}: {e.__cause__}")


**Note:** `load_config()` catches the LOW-level problem (`FileNotFoundError` — a missing file) but re-raises it as a more meaningful, HIGH-level problem for its caller (`RuntimeError("Could not load configuration")`) — the caller shouldn't need to know the config happens to be stored in a file at all. The `from e` is what links them: `e.__cause__` on the new `RuntimeError` still points back to the original `FileNotFoundError`, so nothing about the root cause is lost, even though the caller only ever sees `RuntimeError`.

> ❌ Misconception: `raise NewError(...)` (without `from`) inside an `except` block loses the original exception completely.
> ✅ Correct: Python actually keeps an implicit link too (`__context__`) even without `from` — but `from e` makes it EXPLICIT and intentional, marking `e` specifically as `__cause__`, which shows up clearly in the traceback as "The above exception was the direct cause of the following exception."

---

## 🌳 The Exception Hierarchy

Every exception type in Python is a CLASS, and they're all organized into a family tree rooted at `BaseException`. Knowing this hierarchy explains why `except ArithmeticError:` also catches a `ZeroDivisionError`, and why catching `Exception` (not `BaseException`) is the sensible "catch almost everything" choice.

In [ ]:
print(ZeroDivisionError.__mro__)   # the full chain of ancestor classes

print(issubclass(ZeroDivisionError, ArithmeticError))   # True
print(issubclass(ArithmeticError, Exception))            # True
print(issubclass(Exception, BaseException))               # True


**Note:** `ZeroDivisionError.__mro__` shows the chain: `ZeroDivisionError → ArithmeticError → Exception → BaseException → object`. This is exactly the same **Method Resolution Order** concept used for regular class inheritance — exceptions are just classes, and `except SomeType:` catches that type AND every subclass of it, walking up this same chain. That's why `except ArithmeticError:` would ALSO catch a `ZeroDivisionError`, `OverflowError`, or `FloatingPointError` — they all descend from it.

| Common Exception | Typically Raised When... |
|---|---|
| `ValueError` | A value has the right type, but an invalid value (`int("abc")`) |
| `TypeError` | An operation gets the wrong TYPE entirely (`"a" + 5`) |
| `KeyError` | A dictionary key doesn't exist (`d["missing_key"]`) |
| `IndexError` | A list/sequence index is out of range (`[1, 2, 3][10]`) |
| `ZeroDivisionError` | Dividing (or modulo-ing) by zero |
| `FileNotFoundError` | Opening a file that doesn't exist |
| `AttributeError` | Accessing an attribute/method that doesn't exist on an object |
| `NameError` | Using a variable/function name that was never defined (often a typo) |

---

## ⚖ Comparison

| | Syntax Error | Exception (Runtime Error) |
|---|---|---|
| When it's detected | BEFORE the program even starts running — Python can't parse the code at all | WHILE the program is running, on a specific line, during execution |
| Example | `if x == 5` (missing colon) | `10 / 0` (perfectly valid syntax, fails when it actually runs) |
| Can `try`/`except` catch it? | No — the code never successfully starts running in the first place | Yes — that's exactly what `try`/`except` is for |

| | `except Exception:` | Bare `except:` |
|---|---|---|
| Catches normal errors (`ValueError`, `KeyError`, ...) | ✅ Yes | ✅ Yes |
| Catches `KeyboardInterrupt` (Ctrl+C) / `SystemExit` | ❌ No — these inherit from `BaseException`, not `Exception` | ✅ Yes — accidentally swallows a user's attempt to stop the program |
| Recommended? | ✅ Reasonable as a last resort / logging catch-all | ❌ Almost never — too broad, hides real bugs |

---

## ⚠ Common Misconceptions

❌ `try`/`except` fixes the underlying problem automatically.
✅ It only stops the CRASH — you still have to write the actual response (retry, default value, friendly message). The bug or bad input still exists; you're just choosing how the program reacts to it.

❌ A bare `except:` is a safe, convenient way to "catch everything, just in case."
✅ It catches things you never meant to, like typos (`NameError`) and `KeyboardInterrupt` — hiding real bugs behind a message that looks like an expected failure.

❌ `else` (with `try`) is just a stylistic alternative to putting code at the end of `try`.
✅ `else` specifically means "only if NOTHING in `try` raised an exception" — code directly inside `try` would also be caught by `except` if IT happened to fail, which `else` avoids.

❌ `finally` only runs if the exception was actually caught.
✅ `finally` always runs — success, handled failure, or even an exception that's never caught at all (it still crashes afterward, but `finally` runs first).

❌ Custom exceptions need to override many methods to work correctly.
✅ Inheriting from `Exception` and writing `__init__` (with `super().__init__(message)`) is usually enough — everything else needed to be catchable and printable comes for granted from `Exception`.

❌ `try`/`except` can catch a `SyntaxError` in the code it's wrapping.
✅ A `SyntaxError` means Python couldn't even understand the code enough to start running it — `try`/`except` only catches problems that happen WHILE running valid code, not problems that stop it from running at all.

---

## 🔍 Interview Questions

- What is an exception, and how is it different from a syntax error?
- What happens, step by step, when an exception is raised inside a `try` block that has a matching `except`? What if there's no matching `except`?
- Why is a bare `except:` generally considered bad practice?
- What's the difference between `else` and just writing code at the end of a `try` block?
- When does `finally` run, exactly? Does it run if the exception was never caught at all?
- How do you write and raise your own custom exception? Why would you do that instead of raising a built-in one like `ValueError`?
- What does `raise NewError(...) from original_error` actually do, and why is it useful for debugging?
- Why does `except ArithmeticError:` also catch a `ZeroDivisionError`?
- What's the practical difference between catching `Exception` and catching `BaseException`?

---

## 📝 Quick Revision

- An exception is an unexpected runtime event; unhandled, it crashes the program and prints a traceback.
- `try` holds risky code; `except SomeError as e:` catches a matching exception and lets the program keep running.
- Multiple `except` blocks can catch different exception types differently; a tuple `except (A, B):` catches several types with one handler.
- A bare `except:` catches EVERYTHING, including typos and `KeyboardInterrupt` — almost always avoid it; prefer specific types, or `except Exception:` at most.
- `else` runs only on success (no exception); `finally` always runs, exception or not — the right place for cleanup.
- `raise SomeError("message")` triggers an exception on purpose, from your own code.
- A custom exception is a class inheriting from `Exception`, letting you name domain-specific problems and attach extra data to them.
- `raise NewError(...) from original` chains a new exception to the one that caused it, preserving the real root cause for debugging.
- Every exception is a class, organized in a hierarchy rooted at `BaseException` — catching a parent type catches all its subclasses too.
- Syntax errors are caught before the program runs at all and CANNOT be handled by `try`/`except`; exceptions happen during execution and CAN be.

---

## 🎓 Cheat Sheet

| Concept | One-Line Meaning |
|----------|------------------|
| `try` / `except` | Attempt risky code; catch and handle a matching exception |
| `except (A, B) as e:` | Catch multiple exception types with one handler |
| Bare `except:` | Catches everything — avoid; too broad, hides bugs |
| `else` (with try) | Runs only if `try` succeeded with no exception |
| `finally` | Always runs — success, failure, or uncaught — used for cleanup |
| `raise SomeError("msg")` | Trigger an exception yourself, on purpose |
| Custom exception | `class MyError(Exception):` — names a domain-specific problem |
| `raise New(...) from old` | Chains a new exception to its real root cause |
| `BaseException` | The root of the entire exception hierarchy |
| `Exception` | The base for almost all "normal" catchable errors (excludes `KeyboardInterrupt`/`SystemExit`) |

---

## 📖 Related Topics

Since this topic is **Exception Handling**, next recommended topics:

- [OOP in Python — Part 1](15_Oops_part1.ipynb) and [Part 2](15_Oops_part2.ipynb) (custom exceptions are just classes — inheritance, `super()`, and `isinstance()` all apply directly)
- Context Managers and the `with` statement (a cleaner alternative to manual `try`/`finally` for resource cleanup, like closing files)
- File Handling (`FileNotFoundError`, `PermissionError`, and other I/O-related exceptions)
- Logging (`logging` module — often paired with exception handling for real applications, instead of just `print()`)
- Assertions (`assert`) — a related-but-different tool for catching programmer mistakes during development

---

## ✅ Key Takeaways

1. An exception is a runtime event that disrupts normal program flow; unhandled, it crashes the program and prints a traceback.
2. `try`/`except` lets a program catch a problem and keep running, instead of stopping dead at the first sign of trouble.
3. Catch SPECIFIC exception types whenever possible — a bare `except:` catches everything, including typos and things you never meant to hide.
4. `else` runs only on success; `finally` always runs regardless of outcome — the right place for guaranteed cleanup.
5. `raise` lets you trigger exceptions from your own code, and a custom exception class (inheriting from `Exception`) lets you name domain-specific problems and attach extra data to them.
6. `raise NewError(...) from original_error` preserves the real root cause when one exception happens while handling another.
7. Exceptions are organized in a class hierarchy rooted at `BaseException` — catching a parent type (like `ArithmeticError`) also catches all its subclasses (like `ZeroDivisionError`).